### 9. Genomför en PCA på “car_price_dataset.csv” från kapitel 3 innan du modellerar det med ML. Hur påverkas resultatet?

In [8]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Ladda data
data = pd.read_csv('car_price_dataset.csv', sep=';')

X = data.drop(columns='Price')
y = data['Price']

# Dela upp i tränings- och testdata
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

numeric_features = ['Year', 'Engine_Size', 'Mileage', 'Doors', 'Owner_Count']
categorical_features = ['Brand', 'Model', 'Fuel_Type', 'Transmission']

# Förbehandling: standardisera numeriska variabler och skapa dummy-variabler för kategoriska
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', StandardScaler(), numeric_features),
        ('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
    ],
    remainder='drop'
)

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

print('Antal ursprungliga features efter preprocessing:', X_train_proc.shape[1])

# PCA efter kodning och skalning
pca = PCA(n_components=0.95, random_state=42)
X_train_pca = pca.fit_transform(X_train_proc)
X_test_pca = pca.transform(X_test_proc)

print('Antal PCA-komponenter:', X_train_pca.shape[1])

# Jämför två modeller efter PCA
models = {
    'Ridge': Ridge(alpha=10.0),
    'RandomForest': RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        min_samples_leaf=2,
        n_jobs=-1,
    )
}

for name, model in models.items():
    model.fit(X_train_pca, y_train)
    y_pred = model.predict(X_test_pca)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print(f'\nModel: {name}')
    print(f'MAE:  {mae:.2f}')
    print(f'RMSE: {rmse:.2f}')
    print(f'R2:   {r2:.3f}')

# Jämförelse utan PCA för att se effekten
model_no_pca = Ridge(alpha=10.0)
model_no_pca.fit(X_train_proc, y_train)
y_pred_no_pca = model_no_pca.predict(X_test_proc)

mae_no_pca = mean_absolute_error(y_test, y_pred_no_pca)
rmse_no_pca = np.sqrt(mean_squared_error(y_test, y_pred_no_pca))
r2_no_pca = r2_score(y_test, y_pred_no_pca)

print('\nJämförelse utan PCA (Ridge):')
print(f'MAE:  {mae_no_pca:.2f}')
print(f'RMSE: {rmse_no_pca:.2f}')
print(f'R2:   {r2_no_pca:.3f}')

print('\nTolkning:')
print('- PCA minskar antalet features och kan göra modellen snabbare och mindre känslig för överanpassning.')
print('- Men PCA kan också förlora viktig information, så resultatet kan bli sämre för modeller som RandomForest.')
print('- För Ridge kan PCA ibland ge liknande eller något bättre resultat om det finns mycket korrelation mellan features.')
print('- I detta dataset är det ofta bäst att jämföra både med och utan PCA innan du väljer modell.')

Antal ursprungliga features efter preprocessing: 52
Antal PCA-komponenter: 27

Model: Ridge
MAE:  23.65
RMSE: 65.64
R2:   1.000

Model: RandomForest
MAE:  453.09
RMSE: 579.39
R2:   0.963

Jämförelse utan PCA (Ridge):
MAE:  23.50
RMSE: 65.55
R2:   1.000

Tolkning:
- PCA minskar antalet features och kan göra modellen snabbare och mindre känslig för överanpassning.
- Men PCA kan också förlora viktig information, så resultatet kan bli sämre för modeller som RandomForest.
- För Ridge kan PCA ibland ge liknande eller något bättre resultat om det finns mycket korrelation mellan features.
- I detta dataset är det ofta bäst att jämföra både med och utan PCA innan du väljer modell.


### Hur påverkas resultatet?
#### 1. PCA minskar dimensionerna. Det innebär att modellen får färre inputvariabler. Det kan:
    - göra träningen snabbare.
    - minska överanpassning
    - hjälpa om det finns mycket multikollinearitet


#### 2. Men man kan förlora information. PCA bygger en ny bas av huvudkomponenter. Om viktiga detaljer i data är unika eller inte fångas av de första komponenterna, kan modellens precision bli sämre.



3. För RandomForest blir PCA ofta inte så användbart
Random Forest fungerar ofta bättre på de ursprungliga features, eftersom den kan välja de mest relevanta variablerna direkt. PCA kan därmed göra modellen sämre för beslutsträd-baserade modeller.

4. För Ridge/LinearRegression kan PCA hjälpa mer
Där har PCA ofta större effekt, eftersom linjära modeller kan få problem med många korrelerade feature-värden. Men här måste du fortfarande tänka på att:

PCA gör egenskaperna mindre tolkbara
kategoriska variabler blir “blandade” i komponenter
det blir svårare att förstå varför modellen predikterar ett visst pris



Det finns redan relativt få egenskaper, så PCA är inte alltid nödvändig
Modellen i notebooket använder Ridge och RandomForest
RandomForest brukar ofta prestera bra utan PCA
Ridge kan förbättras något, men inte alltid dramatiskt
Det vanligaste resultatet blir:

samma eller något sämre resultat för Random Forest


ibland bättre eller likvärdigt resultat för Ridge
snabbare träning, men mindre tolkningsbar modell
